In [ ]:
import os
import pandas as pd
import panel as pn
import sqlite3

pn.extension()

In [43]:
# Define the path for the SQLite database
database_path = 'data/crash_data.db'

# Check if the database file exists
if not os.path.exists(database_path):
    print(f"Error: The database file '{database_path}' does not exist.")
else:
    try:
        # Connect to the database
        conn = sqlite3.connect(database_path)

        query = """
                SELECT i.IncidentID, i.County, i.CollisionDate,
                    i.MotorVehiclesInvolved as Vehicles_Involved,
                    i.NumberKilled AS Fatalities,
                    i.NumberInjured AS Injuries, i.Weather,
                    i.RdwyConditionCode AS Rdwy_Condition, i.MannerofCollision,
                    i.RdwyCharacter, i.LightCondition,
                    r.Road_Name, r.Milepoint,
                    r.Speed_Limit_Posted_MPH AS Speed_Limit
                FROM ksp_incidents AS i
                JOIN Roadway_Characteristics_API AS r
                    ON i.IncidentID = r.IncidentID
                WHERE r.Route_Type IN ('I', 'PKWY', 'US', 'KY')
                """

        # Execute the query and fetch the results into a DataFrame
        df = pd.read_sql_query(query, conn)
        print(df)

    except sqlite3.OperationalError as e:
        print(f"OperationalError: {e}")
    finally:
        if conn:
            conn.close()

      IncidentID     County CollisionDate  Vehicles_Involved  Fatalities  \
0       32655798    BULLITT    2023-12-30                  3           0   
1       32660760  JEFFERSON    2023-12-30                  1           0   
2       32660920  JEFFERSON    2023-12-29                  2           0   
3       32646189    BULLITT    2023-12-28                  2           0   
4       32654634     OLDHAM    2023-12-28                  1           0   
...          ...        ...           ...                ...         ...   
3679    32686833     OLDHAM    2024-01-03                  1           0   
3680    32672935    BULLITT    2024-01-02                  2           0   
3681    32676655   MARSHALL    2024-01-02                  2           0   
3682    32658708    BULLITT    2024-01-01                  3           0   
3683    32659228    BULLITT    2024-01-01                  2           0   

      Injuries        Weather Rdwy_Condition             MannerofCollision  \
0        

In [44]:
# build the Panel pane
df_pane = pn.pane.DataFrame(df, width=600)

# df_pane

In [50]:
#pn.panel(df_pane.param, parameters=['bold_rows', 'index', 'header', 'max_rows', 'show_dimensions'],
#         widgets={'max_rows': {'start': 1, 'end': len(df), 'value': len(df)}})

In [46]:
links = pd.DataFrame({
    "Site": ["KSP Crash Data Search", "KYTC Roadway Characterists API",
             "Github", "Twitter"],
    "url": ["http://crashinformationky.org/AdvancedSearch",
            "https://kytc-api-v100-lts-qrntk7e3ra-uc.a.run.app/docs#/Route%20Services/get_route_info_by_coordinates_api_route_GetRouteInfoByCoordinates_get",
            "https://github.com/holoviz/panel",
            "https://twitter.com/Panel_org"]
})
links["value"]="<a href='" + links["url"] + "' target='_blank'>" + links["Site"] + "</a>"
pn.pane.DataFrame(links, escape=False, width=800, index=False)

BokehModel(combine_events=True, render_bundle={'docs_json': {'1a7a260c-2212-46b0-96fc-8f7d38109379': {'version…

In [47]:
table = pn.pane.DataFrame(df.head(50), sizing_mode="stretch_both", max_height=200)
table

BokehModel(combine_events=True, render_bundle={'docs_json': {'799572c6-435a-435c-b27f-8f689e308d00': {'version…

In [49]:
pn.Column("## Incident Data", table, "## Data from KSP & KYTC",
          height=300, width=500).servable()

BokehModel(combine_events=True, render_bundle={'docs_json': {'04f90f5f-12fb-4272-a15f-f4fa143b0895': {'version…